# Notebook 06 — Bayesian Regression & GLMs

## Background

Generalized Linear Models (GLMs) are the backbone of applied statistics: linear regression, logistic regression, Poisson regression, and their extensions cover the vast majority of real-world modeling tasks. In the frequentist tradition you get point estimates and confidence intervals from maximum likelihood. In the Bayesian tradition you get full posterior distributions over coefficients — and with PyMC, you get them for free from `pm.sample()`.

The Bayesian GLM workflow adds three things that the frequentist version lacks:
1. **Uncertainty over coefficients**: not just a single estimate, but a distribution
2. **Posterior predictive distributions**: predictions that properly propagate coefficient uncertainty into forecast uncertainty
3. **Model criticism via PPCs**: check whether the chosen likelihood is appropriate for the data

This notebook builds four models in order of increasing complexity: linear regression (Normal likelihood), logistic regression (Bernoulli), Poisson regression (count data), and negative binomial regression (overdispersed counts).

## What You'll Learn

- Bayesian linear regression: coefficient posteriors, prediction intervals, credible intervals
- Bayesian logistic regression: log-odds, probability scale, decision boundaries
- Bayesian Poisson regression: rate modeling, exposure offset, overdispersion detection
- Negative binomial: when Poisson fails (overdispersion), add a dispersion parameter
- Interpreting GLM coefficients on the natural vs. link-function scale

## Why This Matters

The frequentist GLM gives you a single point estimate. But in practice, you almost always want to know: how uncertain is this estimate? What's the range of plausible effect sizes? For small datasets, the uncertainty is large and ignoring it leads to overconfident decisions. Bayesian GLMs make this uncertainty explicit and propagate it into every downstream use of the model.

In [ ]:
import pymc as pm
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print(f'PyMC {pm.__version__}  |  ArviZ {az.__version__}')

## Part 1 — Bayesian Linear Regression

**Model**: `y_i = alpha + beta * x_i + epsilon_i`, `epsilon_i ~ Normal(0, sigma)`

In matrix form: `y ~ Normal(X @ beta, sigma)`. The Bayesian version puts priors on all parameters and returns a joint posterior over `(alpha, beta, sigma)`.

In [ ]:
np.random.seed(42)

# House price prediction (simplified): sqft, bedrooms, price
n = 80
sqft      = np.random.uniform(800, 3500, n)
bedrooms  = np.random.choice([2, 3, 4, 5], n, p=[0.2, 0.4, 0.3, 0.1])
true_alpha  = 50_000
true_b_sqft = 120      # $120 per sqft
true_b_bed  = 15_000   # $15k per bedroom
true_sigma  = 30_000

price = (true_alpha
         + true_b_sqft * sqft
         + true_b_bed  * bedrooms
         + np.random.normal(0, true_sigma, n))

# Standardize predictors for better sampling geometry
sqft_mean, sqft_std = sqft.mean(), sqft.std()
bed_mean,  bed_std  = bedrooms.mean(), bedrooms.std()
sqft_z   = (sqft - sqft_mean) / sqft_std
bed_z    = (bedrooms - bed_mean) / bed_std

# Price in $1000s for readability
price_k = price / 1000
print(f'House price dataset: n={n}')
print(f'Price range: ${price.min()/1000:.0f}k -- ${price.max()/1000:.0f}k')

with pm.Model() as linear_model:
    # Priors on standardized scale
    alpha    = pm.Normal('alpha',    mu=300, sigma=200)   # intercept in $k
    b_sqft   = pm.Normal('b_sqft',   mu=0,   sigma=200)  # effect of 1-SD sqft
    b_bed    = pm.Normal('b_bed',    mu=0,   sigma=100)  # effect of 1-SD bedrooms
    sigma    = pm.HalfNormal('sigma', sigma=100)
    
    mu = alpha + b_sqft * sqft_z + b_bed * bed_z
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=price_k)
    
    trace_lm = pm.sample(draws=2000, tune=1000, chains=4,
                          random_seed=42, progressbar=True)

print(az.summary(trace_lm, var_names=['alpha', 'b_sqft', 'b_bed', 'sigma']).to_string())

# Convert coefficients back to original scale
b_sqft_orig = trace_lm.posterior['b_sqft'].values.flatten() / sqft_std
b_bed_orig  = trace_lm.posterior['b_bed'].values.flatten()  / bed_std
print(f'\nCoefficients on original scale ($/unit):')
print(f'  sqft effect: ${b_sqft_orig.mean()*1000:.0f} (94% CI: ${np.percentile(b_sqft_orig, 3)*1000:.0f}, ${np.percentile(b_sqft_orig, 97)*1000:.0f}) per sqft')
print(f'  bedroom eff: ${b_bed_orig.mean()*1000:.0f} (94% CI: ${np.percentile(b_bed_orig, 3)*1000:.0f}, ${np.percentile(b_bed_orig, 97)*1000:.0f}) per bedroom')
print(f'  True sqft: ${true_b_sqft}, True bedroom: ${true_b_bed}')

In [ ]:
# Posterior predictive: prediction interval for new homes
new_homes = np.array([
    [1200, 2],
    [2000, 3],
    [3000, 4],
])

alpha_samp  = trace_lm.posterior['alpha'].values.flatten()
b_sqft_samp = trace_lm.posterior['b_sqft'].values.flatten()
b_bed_samp  = trace_lm.posterior['b_bed'].values.flatten()
sigma_samp  = trace_lm.posterior['sigma'].values.flatten()

print(f'Predictions for new homes (price in $k):')
print(f'{"sqft":>6} {"beds":>6} {"pred mean":>11} {"94% PI":>22}')
print('-' * 50)

for sqft_new, beds_new in new_homes:
    sqft_z_new = (sqft_new - sqft_mean) / sqft_std
    bed_z_new  = (beds_new  - bed_mean)  / bed_std
    mu_new     = alpha_samp + b_sqft_samp * sqft_z_new + b_bed_samp * bed_z_new
    # Full predictive: also includes observation noise
    y_new      = np.random.normal(mu_new, sigma_samp)
    lo, hi     = np.percentile(y_new, [3, 97])
    true_y     = (true_alpha + true_b_sqft * sqft_new + true_b_bed * beds_new) / 1000
    print(f'{sqft_new:>6} {beds_new:>6}  ${mu_new.mean():>7.0f}k  (${lo:.0f}k, ${hi:.0f}k)  [true: ${true_y:.0f}k]')

In [ ]:
# Coefficient posterior plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

params = [
    ('b_sqft', b_sqft_orig * 1000, true_b_sqft, '$/sqft'),
    ('b_bed',  b_bed_orig  * 1000, true_b_bed,  '$/bedroom'),
    ('sigma',  trace_lm.posterior['sigma'].values.flatten() * 1000, true_sigma, '$'),
]
for ax, (name, samp, true_val, unit) in zip(axes, params):
    az.plot_posterior({'value': samp[None, None, :]},
                      var_names=['value'], hdi_prob=0.94, ax=ax)
    ax.axvline(true_val, color='red', linestyle='--', lw=1.5, label=f'True: {true_val}')
    ax.set_title(f'{name} ({unit})')
    ax.legend(fontsize=8)

plt.suptitle('Posterior Distributions: Linear Regression Coefficients', fontsize=12)
plt.tight_layout()
plt.show()

## Part 2 — Bayesian Logistic Regression

**Model**: `log(p/(1-p)) = X @ beta` (logit link), `y ~ Bernoulli(sigmoid(X @ beta))`

Coefficients are on the log-odds scale. The natural interpretation:
- `exp(beta_j)` is the odds ratio for a 1-unit increase in `x_j`
- `sigmoid(X @ beta)` is the predicted probability

Key PyMC pattern: use `logit_p=` kwarg to pass the linear predictor directly (numerically more stable than computing `p = sigmoid(...)` then passing `p=`).

In [ ]:
np.random.seed(42)

# Predict customer churn from usage and support tickets
n = 200
usage   = np.random.normal(50, 15, n)      # monthly usage (sessions)
tickets = np.random.poisson(2, n)           # support tickets in 90 days

true_b0     = -1.5
true_b_use  = -0.04   # more usage -> less likely to churn
true_b_tick = 0.6     # more tickets -> more likely to churn

log_odds = true_b0 + true_b_use * usage + true_b_tick * tickets
p_churn  = 1 / (1 + np.exp(-log_odds))
churned  = np.random.binomial(1, p_churn)

# Standardize
usage_z   = (usage   - usage.mean())   / usage.std()
tickets_z = (tickets - tickets.mean()) / tickets.std()

print(f'Churn dataset: n={n}, churn_rate={churned.mean():.1%}')

with pm.Model() as logistic_model:
    b0      = pm.Normal('b0',      mu=0, sigma=2)
    b_use   = pm.Normal('b_use',   mu=0, sigma=2)
    b_tick  = pm.Normal('b_tick',  mu=0, sigma=2)
    
    logit_p = b0 + b_use * usage_z + b_tick * tickets_z
    
    # Derived: probability for each observation
    p_pred  = pm.Deterministic('p_pred', pm.math.sigmoid(logit_p))
    
    y = pm.Bernoulli('y', logit_p=logit_p, observed=churned)
    
    trace_lr = pm.sample(draws=2000, tune=1000, chains=4,
                          random_seed=42, progressbar=False)

print(az.summary(trace_lr, var_names=['b0', 'b_use', 'b_tick']).to_string())

# Odds ratios (on original standardized scale)
b_use_samp  = trace_lr.posterior['b_use'].values.flatten()
b_tick_samp = trace_lr.posterior['b_tick'].values.flatten()
or_use   = np.exp(b_use_samp)
or_tick  = np.exp(b_tick_samp)
print(f'\nOdds Ratios (per 1-SD increase):')
print(f'  Usage OR:   {or_use.mean():.3f} (94% CI: {np.percentile(or_use, 3):.3f}, {np.percentile(or_use, 97):.3f})')
print(f'  Tickets OR: {or_tick.mean():.3f} (94% CI: {np.percentile(or_tick, 3):.3f}, {np.percentile(or_tick, 97):.3f})')

In [ ]:
# Decision boundary and probability contours
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Posterior probability for varying usage and tickets
b0_samp = trace_lr.posterior['b0'].values.flatten()

usage_range   = np.linspace(usage.min(), usage.max(), 50)
tickets_range = np.linspace(0, tickets.max(), 50)
U, T = np.meshgrid(usage_range, tickets_range)

# Standardize grid
U_z = (U - usage.mean()) / usage.std()
T_z = (T - tickets.mean()) / tickets.std()

# Mean posterior probability
logit_grid = b0_samp.mean() + b_use_samp.mean() * U_z + b_tick_samp.mean() * T_z
p_grid     = 1 / (1 + np.exp(-logit_grid))

ax = axes[0]
c = ax.contourf(U, T, p_grid, levels=15, cmap='RdYlGn_r', vmin=0, vmax=1)
plt.colorbar(c, ax=ax, label='P(churn)')
ax.scatter(usage[churned==0], tickets[churned==0], c='green', alpha=0.3, s=20, label='Retained')
ax.scatter(usage[churned==1], tickets[churned==1], c='red',   alpha=0.5, s=25, marker='x', label='Churned')
ax.contour(U, T, p_grid, levels=[0.5], colors='black', linewidths=2)
ax.set_xlabel('Usage (sessions/month)'); ax.set_ylabel('Support tickets (90 days)')
ax.set_title('P(churn) -- posterior mean\nBlack line = 50% decision boundary')
ax.legend(fontsize=8)

# Uncertainty in P(churn) at a specific operating point
ax = axes[1]
scenarios = [
    ('Low-risk', 70, 0),
    ('Medium-risk', 45, 3),
    ('High-risk', 30, 7),
]
colors_s = ['green', 'darkorange', 'red']

for (label, u, t), color in zip(scenarios, colors_s):
    uz = (u - usage.mean()) / usage.std()
    tz = (t - tickets.mean()) / tickets.std()
    p_samp = 1 / (1 + np.exp(-(b0_samp + b_use_samp * uz + b_tick_samp * tz)))
    ax.hist(p_samp, bins=50, alpha=0.5, color=color, density=True, label=f'{label} (u={u}, t={t})')

ax.set_xlabel('P(churn)'); ax.set_ylabel('Density')
ax.set_title('Posterior P(churn) for specific customers\nFull uncertainty propagated')
ax.legend(fontsize=9)

plt.suptitle('Bayesian Logistic Regression: Churn Prediction', fontsize=12)
plt.tight_layout()
plt.show()

## Part 3 — Bayesian Poisson Regression

**Model**: `log(lambda_i) = X @ beta + log(exposure_i)`, `y_i ~ Poisson(lambda_i)`

Poisson regression models count data where the rate is proportional to an exposure (time, area, population, etc.). The log-link ensures `lambda > 0`. The *offset* `log(exposure)` adjusts for different exposure levels without estimating a coefficient for it — it's a fixed correction.

Coefficients in Poisson regression are rate ratios on the exponentiated scale: `exp(beta_j)` is the multiplicative factor change in the event rate per 1-unit increase in `x_j`.

In [ ]:
np.random.seed(42)

# Model: website incidents per day as a function of deploy rate and team size
n = 120
days_monitored = np.random.choice([7, 14, 30], n, p=[0.3, 0.4, 0.3])  # variable exposure
deploys_per_week = np.random.poisson(5, n)
team_size        = np.random.choice([3, 5, 8, 12], n)

true_b0        = -1.0     # log baseline rate per day
true_b_deploy  = 0.08     # log rate ratio per extra deploy/week
true_b_team    = -0.05    # larger team -> fewer incidents

log_lambda = (true_b0
              + true_b_deploy * deploys_per_week
              + true_b_team   * team_size
              + np.log(days_monitored))  # offset for exposure
incidents = np.random.poisson(np.exp(log_lambda))

# Standardize continuous predictors
d_mean, d_std = deploys_per_week.mean(), deploys_per_week.std()
t_mean, t_std = team_size.mean(),        team_size.std()
d_z = (deploys_per_week - d_mean) / d_std
t_z = (team_size         - t_mean) / t_std

print(f'Incidents dataset: n={n}')
print(f'Incidents range: {incidents.min()} -- {incidents.max()}')
print(f'Exposure (days): {days_monitored.min()} -- {days_monitored.max()}')

with pm.Model() as poisson_model:
    b0       = pm.Normal('b0',       mu=0, sigma=2)
    b_deploy = pm.Normal('b_deploy', mu=0, sigma=1)
    b_team   = pm.Normal('b_team',   mu=0, sigma=1)
    
    # log(lambda) = linear predictor + log(exposure)
    log_mu = b0 + b_deploy * d_z + b_team * t_z + np.log(days_monitored)
    y_obs  = pm.Poisson('y_obs', mu=pm.math.exp(log_mu), observed=incidents)
    
    trace_pois = pm.sample(draws=2000, tune=1000, chains=4,
                            random_seed=42, progressbar=False)

print(az.summary(trace_pois, var_names=['b0', 'b_deploy', 'b_team']).to_string())

# Rate ratios
b_d_samp = trace_pois.posterior['b_deploy'].values.flatten()
b_t_samp = trace_pois.posterior['b_team'].values.flatten()
rr_deploy = np.exp(b_d_samp)
rr_team   = np.exp(b_t_samp)
print(f'\nRate Ratios (per 1-SD increase):')
print(f'  Deploy rate RR: {rr_deploy.mean():.3f} (94% CI: {np.percentile(rr_deploy, 3):.3f}, {np.percentile(rr_deploy, 97):.3f})')
print(f'  Team size RR:   {rr_team.mean():.3f} (94% CI: {np.percentile(rr_team, 3):.3f}, {np.percentile(rr_team, 97):.3f})')

In [ ]:
# PPC: check for overdispersion (Poisson assumes mean = variance)
with poisson_model:
    ppc_pois = pm.sample_posterior_predictive(trace_pois, random_seed=42)

y_rep = ppc_pois.posterior_predictive['y_obs'].values.reshape(-1, n)

# Key test: variance/mean ratio (should be ~1 for Poisson)
obs_var_mean_ratio = incidents.var() / incidents.mean()
rep_var_mean_ratio = [y_rep[i].var() / y_rep[i].mean() for i in range(len(y_rep))]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
az.plot_ppc(ppc_pois, observed=True, num_pp_samples=80, ax=ax)
ax.set_title('Poisson PPC')

ax = axes[1]
ax.hist(rep_var_mean_ratio, bins=40, color='steelblue', alpha=0.7, edgecolor='white', density=True)
ax.axvline(1.0, color='black', linestyle='--', lw=1.5, label='Poisson: var=mean')
ax.axvline(obs_var_mean_ratio, color='red', lw=2, linestyle='--',
           label=f'Observed: {obs_var_mean_ratio:.2f}')
ax.set_xlabel('Variance / Mean'); ax.set_ylabel('Density')
ax.set_title('Overdispersion Check\nVar/Mean ratio')
ax.legend()

plt.suptitle('Poisson Model PPC: Detecting Overdispersion', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Observed variance/mean: {obs_var_mean_ratio:.2f}')
print(f'Replicated mean: {np.mean(rep_var_mean_ratio):.2f}')
print('If observed ratio >> 1.0, Poisson is underdispersed -> use NegBin')

## Part 4 — Negative Binomial: Robust Count Regression

Poisson constrains variance = mean. In real count data this almost never holds — variance is usually larger (overdispersion). The Negative Binomial distribution relaxes this with a dispersion parameter `alpha` (or equivalently `phi = 1/alpha`):

```
y ~ NegBin(mu, alpha)  where  Var(y) = mu + mu^2 / alpha
```

As `alpha -> infinity`, the NegBin converges to Poisson. Small `alpha` means high overdispersion. Making `alpha` a parameter lets the model learn the dispersion from data.

In [ ]:
# Generate overdispersed count data
np.random.seed(42)
n_nb = 150
x_nb = np.random.uniform(0, 5, n_nb)
true_alpha_nb = 2.0   # dispersion; smaller = more overdispersed
true_mu_nb = np.exp(0.5 + 0.4 * x_nb)

# NegBin: draw from Gamma-Poisson mixture
# p = alpha / (alpha + mu) for scipy parameterization
counts_nb = np.array([
    stats.nbinom.rvs(true_alpha_nb, true_alpha_nb / (true_alpha_nb + mu))
    for mu in true_mu_nb
])

print(f'Count data: mean={counts_nb.mean():.1f}, var={counts_nb.var():.1f}')
print(f'Var/mean ratio: {counts_nb.var()/counts_nb.mean():.2f} (>1 = overdispersed)')

# Fit Poisson and NegBin -- compare
x_z_nb = (x_nb - x_nb.mean()) / x_nb.std()

with pm.Model() as negbin_model:
    b0    = pm.Normal('b0',    mu=0, sigma=3)
    b1    = pm.Normal('b1',    mu=0, sigma=2)
    alpha = pm.Exponential('alpha', lam=1)  # dispersion parameter
    
    mu_pred = pm.math.exp(b0 + b1 * x_z_nb)
    y_obs   = pm.NegativeBinomial('y_obs', mu=mu_pred, alpha=alpha, observed=counts_nb)
    
    trace_nb = pm.sample(draws=2000, tune=1000, chains=4,
                          random_seed=42, progressbar=False)
    ppc_nb   = pm.sample_posterior_predictive(trace_nb, random_seed=42)

print('\nNegative Binomial summary:')
print(az.summary(trace_nb, var_names=['b0', 'b1', 'alpha']).to_string())
print(f'\nTrue alpha: {true_alpha_nb}')

alpha_samp = trace_nb.posterior['alpha'].values.flatten()
print(f'Posterior alpha: {alpha_samp.mean():.2f} (94% CI: {np.percentile(alpha_samp, 3):.2f}, {np.percentile(alpha_samp, 97):.2f})')

In [ ]:
# Compare Poisson vs NegBin fit on the same overdispersed data
with pm.Model() as pois_on_nb_data:
    b0  = pm.Normal('b0', mu=0, sigma=3)
    b1  = pm.Normal('b1', mu=0, sigma=2)
    mu  = pm.math.exp(b0 + b1 * x_z_nb)
    y   = pm.Poisson('y', mu=mu, observed=counts_nb)
    trace_pois_nb = pm.sample(draws=500, tune=500, chains=2,
                               random_seed=42, progressbar=False)
    ppc_pois_nb   = pm.sample_posterior_predictive(trace_pois_nb, random_seed=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (ppc, title, color) in zip(axes, [
    (ppc_pois_nb, 'Poisson (underdispersed)', 'darkorange'),
    (ppc_nb,      'Negative Binomial (correct)', 'steelblue'),
]):
    var_key = 'y' if 'y' in ppc.posterior_predictive else 'y_obs'
    y_rep_plot = ppc.posterior_predictive[var_key].values.reshape(-1, n_nb)
    # Sample of replicated histograms
    max_count = max(counts_nb.max(), y_rep_plot.max())
    bins = np.arange(0, max_count + 2)
    for i in range(50):
        hist, _ = np.histogram(y_rep_plot[i], bins=bins, density=True)
        ax.step(bins[:-1], hist, where='post', alpha=0.05, color=color)
    hist_obs, _ = np.histogram(counts_nb, bins=bins, density=True)
    ax.step(bins[:-1], hist_obs, where='post', color='black', lw=2, label='Observed')
    ax.set_xlabel('Count'); ax.set_ylabel('Density')
    ax.set_title(title); ax.legend()

plt.suptitle('PPC: Poisson vs Negative Binomial on Overdispersed Counts', fontsize=12)
plt.tight_layout()
plt.show()

## Part 5 — GLM Selection Guide

Choosing the right GLM depends on the outcome type and its properties:

In [ ]:
print('Bayesian GLM Quick Reference')
print('=' * 70)
models_table = [
    ('Linear',            'Continuous, unbounded',  'None (identity)',  'Normal',          'sigma'),
    ('Logistic',          'Binary 0/1',             'logit',            'Bernoulli',       'none'),
    ('Poisson',           'Counts, mean = var',     'log',              'Poisson',         'none'),
    ('Neg. Binomial',     'Counts, var > mean',     'log',              'NegativeBinomial','alpha (dispersion)'),
    ('Beta regression',   'Proportions (0,1)',       'logit',            'Beta',            'kappa (precision)'),
    ('Ordered logistic',  'Ordinal categories',      'logit (cutpoints)','OrderedLogistic', 'cutpoints'),
]
print(f'{"Model":20} {"Outcome":25} {"Link":16} {"Likelihood":18} {"Extra param":20}')
print('-' * 103)
for row in models_table:
    print(f'{row[0]:20} {row[1]:25} {row[2]:16} {row[3]:18} {row[4]:20}')

print()
print('PyMC likelihood syntax:')
print('  Normal:           pm.Normal("y", mu=mu, sigma=sigma, observed=y)')
print('  Bernoulli:        pm.Bernoulli("y", logit_p=logit_p, observed=y)')
print('  Poisson:          pm.Poisson("y", mu=exp(log_mu), observed=y)')
print('  NegativeBinomial: pm.NegativeBinomial("y", mu=mu, alpha=alpha, observed=y)')
print('  StudentT:         pm.StudentT("y", nu=nu, mu=mu, sigma=sigma, observed=y)')

## Summary

Bayesian GLMs give you everything frequentist GLMs do, plus:

- **Full posterior** over all coefficients — not just a point estimate
- **Posterior predictive intervals** that propagate coefficient uncertainty into forecasts
- **PPCs** to diagnose misspecification (overdispersion, wrong link, missing predictors)

**Key patterns**:
- Standardize predictors before modeling — better sampling geometry, comparable coefficient magnitudes
- Use `logit_p=` for Bernoulli (numerical stability over `p=sigmoid(...)`)
- Always add `log(exposure)` offset for Poisson rate models
- If Poisson PPC shows overdispersion → switch to NegativeBinomial
- `pm.Deterministic` to track odds ratios, rate ratios, or probabilities in the trace

**Next**: Notebook 07 — Model Comparison. WAIC, LOO-CV, Bayes factors. When do you have enough evidence to prefer one model over another?